In [0]:
bronze_schemapath  = "`bronze_catalog`.default"
silver_schemapath = "`silver_catalog`.default"

In [0]:
tables = spark.catalog.listTables(dbName=bronze_schemapath)

In [0]:
fivetran_ingested_column_to_remove = [
    "_file", 	
    "_fivetran_synced", 
    "_modified",
    "_line"
]

In [0]:
def remove_fivetran_ingested_column(df):
    for column in fivetran_ingested_column_to_remove:
        df = df.drop(column)
    return df

In [0]:
# Read from bronze, remove metadata columns, and write to silver
df = spark.read.table(f"{bronze_schemapath}.customer_table")
df_clean = remove_fivetran_ingested_column(df)
#df_clean.write.format("delta").mode("overwrite").saveAsTable(f"{silver_schemapath}.customer_table")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

def fill_id_column(df, column_name):
    w = Window().orderBy(df.columns[0])
    df = df.withColumn(column_name, row_number().over(w))
    return df

In [0]:
from pyspark.sql.functions import to_date, col, split
df = spark.read.table(f"{silver_schemapath}.customer_table")
df = df.withColumn("account_created_date", to_date(col("account_created_date"), "dd-MM-yyyy"))
df = df.dropDuplicates()
df = df.withColumn(
    "customer_internal_id",
    split(col("customer_name"), " ")[1]
)
display(df)